# TP4 — MQTT and HTTP inference

Run all and upload the included models/LR-Pipeline.pkl and XGB-Pipeline.pkl when prompted. Supporting Python modules are recreated by the notebook. The local demo runs in Colab; a network subscriber should run on your computer next to a reachable broker.

This notebook contains the full Python solution. Board projects also require the included firmware. Read EXPLICATION_AR.md for the detailed Arabic explanation.

[Assignment](https://github.com/ahmed3991/AIoT-2025/tree/main/TP4)

In [1]:
# Install missing dependencies before importing the model libraries.
import importlib.util
import importlib.metadata
import subprocess
import sys

required = {'pandas': 'pandas==2.2.3', 'sklearn': 'scikit-learn==1.8.0', 'xgboost': 'xgboost==3.4.1', 'paho.mqtt': 'paho-mqtt==2.1.0', 'flask': 'Flask'}
missing = []
for module_name, package_spec in required.items():
    try:
        found = importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        found = False
    if not found:
        missing.append(package_spec)
    elif '==' in package_spec:
        distribution, expected = package_spec.split('==')
        if importlib.metadata.version(distribution) != expected:
            missing.append(package_spec)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready.')


Dependencies ready.


In [2]:
# Recreate the small support modules so this notebook can run on Colab.
from pathlib import Path
import sys
Path('ai_logic').mkdir(exist_ok=True)
Path('ai_logic\\__init__.py').write_text('"""TP4 inference and transport modules."""\n', encoding='utf-8')
Path('ai_logic\\http_server.py').write_text('"""HTTP alternative: python -m ai_logic.http_server"""\nimport os\nfrom flask import Flask, jsonify, request\nfrom ai_logic.inference import infer\n\napp = Flask(__name__)\napp.config["MAX_CONTENT_LENGTH"] = 16 * 1024\n\n\n@app.post("/infer")\ndef inference_endpoint():\n    try:\n        payload = request.get_json(silent=True)\n        selected = request.args.get("model", "lr")\n        return jsonify(infer(payload, selected))\n    except (ValueError, TypeError, KeyError) as error:\n        return jsonify({"error": str(error)}), 400\n\n\nif __name__ == "__main__":\n    app.run(host=os.environ.get("HTTP_HOST", "127.0.0.1"), port=5000, debug=False)\n', encoding='utf-8')
Path('ai_logic\\inference.py').write_text('"""Shared prediction code for MQTT, HTTP, and the notebook demo."""\nimport math\nimport pickle\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nROOT = Path(__file__).resolve().parents[1]\nFEATURES = ["Temperature[C]", "Humidity[%]", "TVOC[ppb]", "eCO2[ppm]",\n            "Raw H2", "Raw Ethanol", "Pressure[hPa]", "PM1.0", "PM2.5",\n            "NC0.5", "NC1.0", "NC2.5"]\nPIPELINES = {}\n\n\ndef load_models():\n    if PIPELINES:\n        return PIPELINES\n    missing = [f"{name}-Pipeline.pkl" for name in ["LR", "XGB"]\n               if not (ROOT / "models" / f"{name}-Pipeline.pkl").exists()]\n    if missing:\n        try:\n            from google.colab import files\n        except ImportError as error:\n            raise FileNotFoundError("Copy both TP2 pipeline files into models/.") from error\n        print("Upload TP2\'s LR-Pipeline.pkl and XGB-Pipeline.pkl")\n        (ROOT / "models").mkdir(exist_ok=True)\n        for name, content in files.upload().items():\n            if Path(name).name in ["LR-Pipeline.pkl", "XGB-Pipeline.pkl"]:\n                (ROOT / "models" / Path(name).name).write_bytes(content)\n    for name, prefix in [("lr", "LR"), ("xgb", "XGB")]:\n        with (ROOT / "models" / f"{prefix}-Pipeline.pkl").open("rb") as file:\n            PIPELINES[name] = pickle.load(file)\n        if list(PIPELINES[name].feature_names_in_) != FEATURES:\n            raise ValueError("Pipeline sensor order does not match this TP4 schema.")\n    return PIPELINES\n\n\ndef infer(payload, selected="lr"):\n    if selected not in ["lr", "xgb"]:\n        raise ValueError("Choose lr or xgb.")\n    if not isinstance(payload, dict) or not isinstance(payload.get("device_id"), str):\n        raise ValueError("A JSON object with device_id is required.")\n    device_id = payload["device_id"]\n    if not device_id or len(device_id) > 64:\n        raise ValueError("device_id must contain 1–64 characters.")\n    readings = payload.get("readings")\n    if not isinstance(readings, dict) or any(name not in readings for name in FEATURES):\n        raise ValueError("readings must contain all 12 named TP2 sensor values.")\n    values = [readings[name] for name in FEATURES]\n    if any(isinstance(value, bool) or not isinstance(value, (int, float))\n           or not math.isfinite(value) for value in values):\n        raise ValueError("Sensor values must be finite numbers.")\n    frame = pd.DataFrame([values], columns=FEATURES)\n    start = time.perf_counter()\n    decisions = {}\n    for name, pipeline in load_models().items():\n        decisions[name] = {"prediction": int(pipeline.predict(frame)[0]),\n                           "probability": float(pipeline.predict_proba(frame)[0, 1])}\n    return {\n        "device_id": device_id, "request_id": payload.get("request_id", 0),\n        "model": selected, **decisions[selected], "both_models": decisions,\n        "timestamp": time.time(), "server_inference_ms": (time.perf_counter() - start) * 1000,\n    }\n', encoding='utf-8')
Path('ai_logic\\mqtt_ai_subscriber.py').write_text('"""Run: python -m ai_logic.mqtt_ai_subscriber --model lr --broker localhost."""\nimport argparse\nimport json\nimport logging\nimport os\n\nimport paho.mqtt.client as mqtt\nfrom ai_logic.inference import infer, load_models\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", choices=["lr", "xgb"], default="lr")\n    parser.add_argument("--broker", default="localhost")\n    parser.add_argument("--port", type=int, default=1883)\n    parser.add_argument("--tls", action="store_true")\n    args = parser.parse_args()\n    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")\n    load_models()\n\n    def on_connect(client, userdata, flags, reason_code, properties):\n        if reason_code.is_failure:\n            logging.error("Connection rejected: %s", reason_code)\n            return\n        client.subscribe("esp32/data", qos=1)\n        logging.info("Connected; subscribed to esp32/data")\n\n    def on_message(client, userdata, message):\n        try:\n            payload = json.loads(message.payload)\n            response = infer(payload, args.model)\n            client.publish("esp32/control", json.dumps(response), qos=1, retain=False)\n            logging.info("%s", response)\n        except (ValueError, TypeError, KeyError, UnicodeDecodeError) as error:\n            logging.warning("Ignored malformed message: %s", error)\n\n    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="ghanabzia-tp4-ai")\n    client.on_connect = on_connect\n    client.on_message = on_message\n    client.reconnect_delay_set(min_delay=1, max_delay=30)\n    if args.tls:\n        client.tls_set()\n    username = os.environ.get("MQTT_USERNAME")\n    if username:\n        client.username_pw_set(username, os.environ.get("MQTT_PASSWORD"))\n    client.connect_async(args.broker, args.port, keepalive=30)\n    try:\n        client.loop_forever(retry_first_connection=True)\n    except KeyboardInterrupt:\n        client.disconnect()\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
sys.path.insert(0, str(Path.cwd()))


## 1. Imports and models

In [3]:
"""TP4 notebook/local demo: real inference and HTTP without a broker or board."""
import json
from pathlib import Path
from time import perf_counter

from ai_logic.inference import FEATURES, infer, load_models
from ai_logic.http_server import app

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
load_models()



## 2. A valid payload containing all twelve actual TP2 sensor fields

In [4]:
# Replay the first real CSV reading; this is a host demonstration, not a live sensor.
payload = {
    "device_id": "esp-01", "request_id": 1, "timestamp": 0,
    "readings": dict(zip(FEATURES, [20.0, 57.36, 0, 400, 12306, 18520,
                                  939.735, 0, 0, 0, 0, 0])),
}
(ROOT / "sample_payload.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
response = infer(payload, selected="lr")
print("Input:\n", json.dumps(payload, indent=2))
print("Both model predictions:\n", json.dumps(response, indent=2))



Input:
 {
  "device_id": "esp-01",
  "request_id": 1,
  "timestamp": 0,
  "readings": {
    "Temperature[C]": 20.0,
    "Humidity[%]": 57.36,
    "TVOC[ppb]": 0,
    "eCO2[ppm]": 400,
    "Raw H2": 12306,
    "Raw Ethanol": 18520,
    "Pressure[hPa]": 939.735,
    "PM1.0": 0,
    "PM2.5": 0,
    "NC0.5": 0,
    "NC1.0": 0,
    "NC2.5": 0
  }
}
Both model predictions:
 {
  "device_id": "esp-01",
  "request_id": 1,
  "model": "lr",
  "prediction": 1,
  "probability": 0.9999765936236253,
  "both_models": {
    "lr": {
      "prediction": 1,
      "probability": 0.9999765936236253
    },
    "xgb": {
      "prediction": 0,
      "probability": 0.00033970572985708714
    }
  },
  "timestamp": 1791813651.9927256,
  "server_inference_ms": 5.517800000234274
}


## 3. Exercise the real POST endpoint and reject malformed sensor payloads

In [5]:
with app.test_client() as client:
    start = perf_counter()
    http_response = client.post("/infer?model=xgb", json=payload)
    http_local_ms = (perf_counter() - start) * 1000
    assert http_response.status_code == 200
    assert http_response.get_json()["model"] == "xgb"
    assert http_response.get_json()["prediction"] == response["both_models"]["xgb"]["prediction"]
    invalid = client.post("/infer", json={"device_id": "esp-01", "readings": {}})
    assert invalid.status_code == 400
    print("HTTP valid status:", http_response.status_code)
    print("HTTP malformed status:", invalid.status_code)
    print("Local in-process HTTP round trip (ms):", http_local_ms)



HTTP valid status: 200
HTTP malformed status: 400
Local in-process HTTP round trip (ms): 7.219399998575682


## 4. Report and how to run the two network services

In [6]:
(OUT / "demo_response.json").write_text(json.dumps(response, indent=2), encoding="utf-8")
report = f"""# TP4 report

Both TP2 pipelines load successfully with all 12 sensor fields in the original
order. The first dataset reading produces LR probability
{response['both_models']['lr']['probability']:.6f} and XGBoost probability
{response['both_models']['xgb']['probability']:.6f}. Both models are evaluated;
the selected CLI model controls the returned decision. Actual pipeline file sizes:
LR {(ROOT/'models'/'LR-Pipeline.pkl').stat().st_size / 1024:.3f} KB;
XGB {(ROOT/'models'/'XGB-Pipeline.pkl').stat().st_size / 1024:.3f} KB.

The real Flask endpoint was exercised in-process: valid request HTTP 200,
missing sensor fields HTTP 400. The local request round trip was
{http_local_ms:.3f} ms. This is a **host/in-process** measurement, not WiFi/MQTT
or ESP32 end-to-end latency. Firmware echoes request IDs and measures millis()
from publish to response on the same board, avoiding unsynchronized clock subtraction.
No board/network latency is invented.

MQTT: ESP32 -> esp32/data -> Python -> esp32/control -> ESP32 LED.
The subscriber uses QoS 1 for subscribe/reply, reconnect delay 1–30 seconds,
device IDs, finite-number validation and logging. The ESP32 PubSubClient
publisher sends QoS 0 (a library limitation); the JSON includes a request ID so
replies can be matched. QoS 1 can deliver duplicates; repeating LED state is
idempotent. The minimal classroom setup uses a local broker. Python supports
TLS and environment-based username/password for a configured authenticated broker.

HTTP: ESP32 POST /infer -> Python inference -> JSON HTTP response -> LED.
MQTT decouples producers/consumers and supports multiple subscribers, while HTTP
is simple request/response with larger header overhead. CoAP uses a lightweight
REST-style UDP transport; WebSockets provide persistent bidirectional communication.
The default firmware reads DHT22 temperature/humidity and replays fixed demo
values for the other ten sensors. Those constants are explicitly a classroom
simulation; actual deployment must connect all expected sensors or retrain a
two-feature pipeline. It never silently substitutes zeros for missing JSON fields.

Use a locally reachable broker, or the documented public demonstration mode.
For real hardware set WiFi settings and the host's LAN address; for a local
HTTP server accessible on LAN set HTTP_HOST=0.0.0.0. Colab can demonstrate
inference and the endpoint, but it is not a directly reachable local LAN broker.

Source: https://github.com/ahmed3991/AIoT-2025/blob/main/TP4/TP4.md
"""
(OUT / "REPORT.md").write_text(report, encoding="utf-8")
print("Saved results. Network commands (run in separate terminals):")
print("python -m ai_logic.mqtt_ai_subscriber --model lr --broker localhost")
print("python -m ai_logic.http_server")


Saved results. Network commands (run in separate terminals):
python -m ai_logic.mqtt_ai_subscriber --model lr --broker localhost
python -m ai_logic.http_server


In [7]:
from IPython.display import display, Markdown
display(Markdown((OUT).joinpath('REPORT.md').read_text(encoding='utf-8')))


# TP4 report

Both TP2 pipelines load successfully with all 12 sensor fields in the original
order. The first dataset reading produces LR probability
0.999977 and XGBoost probability
0.000340. Both models are evaluated;
the selected CLI model controls the returned decision. Actual pipeline file sizes:
LR 1.642 KB;
XGB 120.317 KB.

The real Flask endpoint was exercised in-process: valid request HTTP 200,
missing sensor fields HTTP 400. The local request round trip was
7.219 ms. This is a **host/in-process** measurement, not WiFi/MQTT
or ESP32 end-to-end latency. Firmware echoes request IDs and measures millis()
from publish to response on the same board, avoiding unsynchronized clock subtraction.
No board/network latency is invented.

MQTT: ESP32 -> esp32/data -> Python -> esp32/control -> ESP32 LED.
The subscriber uses QoS 1 for subscribe/reply, reconnect delay 1–30 seconds,
device IDs, finite-number validation and logging. The ESP32 PubSubClient
publisher sends QoS 0 (a library limitation); the JSON includes a request ID so
replies can be matched. QoS 1 can deliver duplicates; repeating LED state is
idempotent. The minimal classroom setup uses a local broker. Python supports
TLS and environment-based username/password for a configured authenticated broker.

HTTP: ESP32 POST /infer -> Python inference -> JSON HTTP response -> LED.
MQTT decouples producers/consumers and supports multiple subscribers, while HTTP
is simple request/response with larger header overhead. CoAP uses a lightweight
REST-style UDP transport; WebSockets provide persistent bidirectional communication.
The default firmware reads DHT22 temperature/humidity and replays fixed demo
values for the other ten sensors. Those constants are explicitly a classroom
simulation; actual deployment must connect all expected sensors or retrain a
two-feature pipeline. It never silently substitutes zeros for missing JSON fields.

Use a locally reachable broker, or the documented public demonstration mode.
For real hardware set WiFi settings and the host's LAN address; for a local
HTTP server accessible on LAN set HTTP_HOST=0.0.0.0. Colab can demonstrate
inference and the endpoint, but it is not a directly reachable local LAN broker.

Source: https://github.com/ahmed3991/AIoT-2025/blob/main/TP4/TP4.md
